# Day 11 — Final Analysis & Portfolio Visuals

This notebook consolidates the project's validated sales, customer, RFM and segmentation work into a portfolio-ready analytical view. It rebuilds the pipeline from the public UCI Online Retail dataset at runtime, so every KPI, chart and segment result is generated from the actual source data.

No numeric result, ranking or business conclusion is hard-coded.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
from ucimlrepo import fetch_ucirepo

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT))
from src.eda import positive_sales_view, sales_kpis, product_performance
from src.rfm_analysis import build_rfm
from src.clustering_prep import prepare_clustering_features
from src.kmeans_clustering import evaluate_k_range, fit_kmeans
from src.segment_interpretation import profile_clusters, add_relative_segment_labels
from src.final_analysis import monthly_sales_summary, country_summary, customer_concentration, plot_monthly_revenue, plot_segment_revenue, plot_revenue_concentration

online_retail = fetch_ucirepo(id=352)
raw = online_retail.data.features.copy()
raw.columns = [c.strip().lower().replace(' ', '_') for c in raw.columns]
raw['invoice_date'] = pd.to_datetime(raw['invoice_date'], errors='coerce')
for col in ['quantity', 'unit_price', 'customer_id']:
    raw[col] = pd.to_numeric(raw[col], errors='coerce')
clean = raw.drop_duplicates().copy()
clean['revenue'] = clean['quantity'] * clean['unit_price']
sales = positive_sales_view(clean)
print(f'Positive-sales rows: {len(sales):,}')
print(f'KPI snapshot: {sales_kpis(sales)}')

## 1. Executive KPI snapshot

These values are calculated at runtime from positive-price, positive-quantity transactions. The project's cleaning rules intentionally keep cancellations and non-sales rows in the cleaned dataset; this analytical view is used only for sales performance and customer-value analysis.

In [ ]:
kpis = sales_kpis(sales)
pd.DataFrame([kpis])

## 2. Revenue trend

In [ ]:
monthly = monthly_sales_summary(sales)
monthly

In [ ]:
fig, ax = plot_monthly_revenue(monthly)
plt.show()

## 3. Geographic and product performance

In [ ]:
countries = country_summary(sales)
countries.head(10)

In [ ]:
products = product_performance(sales, top_n=10)
products

## 4. Customer value concentration

Ranking customers by observed revenue helps show whether sales are broadly distributed or concentrated among a smaller set of customers. This is descriptive, not a statement about future retention risk.

In [ ]:
concentration = customer_concentration(sales)
concentration.head(10)

In [ ]:
fig, ax = plot_revenue_concentration(concentration)
plt.show()

## 5. Final customer segmentation

The final segmentation reuses the Day 7–10 methodology: RFM is built from positive sales, skewed features are transformed conditionally, features are standardized, k is selected by the highest silhouette score over k=2..8, and cluster IDs are translated into relative business archetypes.

In [ ]:
rfm = build_rfm(sales)
_, scaled_features, skewness = prepare_clustering_features(rfm)
evaluation = evaluate_k_range(scaled_features, range(2, 9), random_state=42, n_init=20)
selected_k = int(evaluation.loc[evaluation['silhouette_score'].idxmax(), 'k'])
model, labels = fit_kmeans(scaled_features, selected_k, random_state=42, n_init=20)
clustered_rfm = rfm.loc[scaled_features.index].copy()
clustered_rfm['cluster'] = labels
profile = add_relative_segment_labels(profile_clusters(clustered_rfm))
print(f'Selected k: {selected_k}')
print(f'Customers segmented: {len(clustered_rfm):,}')
profile

In [ ]:
fig, ax = plot_segment_revenue(profile)
plt.show()

## 6. Portfolio takeaways generated from the analysis

Use the runtime tables and charts above to write the final business narrative. The project should distinguish clearly between observed evidence and recommendations. Recommended actions should remain tied to the relative segment profiles: protect high-value engaged customers, nurture developing customers, improve basket value for frequent lower-value customers, and test win-back activity for high-value customers whose recency has deteriorated.

**Important:** these are strategic hypotheses, not measured campaign effects. The dataset does not contain campaign exposure, margin, marketing cost or retention-outcome fields needed to estimate incremental lift.

## Portfolio presentation checklist

- Lead with the KPI snapshot and revenue trend.
- Use the segment profile to connect analytics to customer actions.
- Keep charts labeled with business-friendly titles and units.
- Do not publish hard-coded numbers unless they came from an executed run of this notebook.
- Do not imply causality or campaign lift from observational transaction data.
- Power BI dashboard construction is reserved for Day 12.